# Dataset 2 Inspection

This notebook inspects the second dataset before preprocessing.

We check the dataset structure, class distribution, image-label matching,
and image dimensions without modifying the original dataset.


# 1. Dataset Structure

In [1]:
from pathlib import Path
import pandas as pd

# Project paths
PROJECT_ROOT = Path(r"C:\Users\DELL\Downloads\mlproject\bone-cancer-detection")
DATASET_ROOT = PROJECT_ROOT / "data" / "raw" / "Dataset"

# Check that Dataset 2 exists
print("Dataset 2 path:", DATASET_ROOT)
print("Dataset exists:", DATASET_ROOT.exists())

# Show the dataset structure
print("\nDataset contents:")
for item in DATASET_ROOT.iterdir():
    print("-", item.name)

Dataset 2 path: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\raw\Dataset
Dataset exists: True

Dataset contents:
- test
- train
- valid


# 2. Label Inspection

In [2]:
# Check each dataset split and its class labels

splits = ["train", "valid", "test"]

for split in splits:
    split_dir = DATASET_ROOT / split
    
    print(f"\n{'=' * 50}")
    print(f"{split.upper()}")
    print(f"{'=' * 50}")
    
    # Find the CSV containing the class labels
    csv_files = list(split_dir.glob("*.csv"))
    
    print("CSV files:", [f.name for f in csv_files])
    
    # Count image files
    image_files = [
        f for f in split_dir.iterdir()
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]
    
    print("Images:", len(image_files))
    
    # Inspect the class CSV
    if csv_files:
        labels = pd.read_csv(csv_files[0])
        
        print("CSV shape:", labels.shape)
        print("Columns:", list(labels.columns))
        display(labels.head())


TRAIN
CSV files: ['_classes.csv']
Images: 7057
CSV shape: (7057, 3)
Columns: ['filename', ' cancer', ' normal']


,filename,cancer,normal
0,-53-_jpg.rf.08d303d0b43c7b71c3581ec02d7390c3.jpg,0,1
1,image-no531-normal-_png.rf.08eeb57b24a9062d798...,0,1
2,IMG0000240_jpg.rf.094924a72ca1ab5e26477e50abac...,0,1
3,bone-cancer_train_2_821_png.rf.094c60132af2cb3...,1,0
4,image-no52-normal-_png.rf.08f33a992911794d2381...,0,1



VALID
CSV files: ['_classes.csv']
Images: 882
CSV shape: (882, 3)
Columns: ['filename', ' cancer', ' normal']


,filename,cancer,normal
0,177_JPG_jpg.rf.016b13419907ef52e805cde43d35225...,1,0
1,IMG0000345_jpg.rf.006fa65fdaa1df5d6287aefdf9ef...,0,1
2,foot_35_1_png.rf.006b991a599ebee5cc6d7bd8678bb...,0,1
3,IMG0000291_jpg.rf.017794ae2367a87da945a33c8fa6...,0,1
4,IMG0000833_jpg.rf.02e4f31bc0df6ad5663194c15508...,0,1



TEST
CSV files: ['_classes.csv']
Images: 872
CSV shape: (872, 3)
Columns: ['filename', ' cancer', ' normal']


,filename,cancer,normal
0,Picture6_jpg.rf.034f47366f61e0cbec31d48d7b4167...,0,1
1,IMG0000041_jpg.rf.0707545fd366027c4732ccf7bbe3...,0,1
2,bone-cancer_train_2_1674_png.rf.08de2e2c775f6f...,1,0
3,bone-cancer_train_2_1873_png.rf.05b25c9da2a561...,1,0
4,chest_45_male_png.rf.0198bd97436b808cc6fbd6cb1...,0,1


In [3]:
# Check the number of cancer and normal images in each split

for split in ["train", "valid", "test"]:
    split_dir = DATASET_ROOT / split
    csv_path = split_dir / "_classes.csv"

    labels = pd.read_csv(csv_path)

    # Remove accidental spaces from column names
    labels.columns = labels.columns.str.strip()

    print(f"\n{'=' * 50}")
    print(split.upper())
    print("=" * 50)

    print("Cancer:", labels["cancer"].sum())
    print("Normal:", labels["normal"].sum())

    print("\nClass distribution:")
    print(labels[["cancer", "normal"]].sum())


TRAIN
Cancer: 3081
Normal: 3976

Class distribution:
cancer    3081
normal    3976
dtype: int64

VALID
Cancer: 398
Normal: 484

Class distribution:
cancer    398
normal    484
dtype: int64

TEST
Cancer: 384
Normal: 488

Class distribution:
cancer    384
normal    488
dtype: int64


# 3. Image Inspection

We inspect the images before preprocessing.

This includes image dimensions, file formats, and aspect ratios.

The original Dataset 2 files are not modified.

In [7]:
# Inspect image dimensions and formats for each dataset split

from PIL import Image
from collections import Counter

for split in ["train", "valid", "test"]:
    
    split_dir = DATASET_ROOT / split
    
    image_files = [
        f for f in split_dir.iterdir()
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]
    
    sizes = []
    formats = []
    
    for image_path in image_files:
        try:
            with Image.open(image_path) as img:
                sizes.append(img.size)
                formats.append(img.format)
        except:
            pass
    
    print(f"\n{'=' * 50}")
    print(split.upper())
    print("=" * 50)
    
    print("Images checked:", len(sizes))
    print("Unique image sizes:", len(set(sizes)))
    print("Image formats:", Counter(formats))
    
    if sizes:
        print("First image size:", sizes[0])


TRAIN
Images checked: 7056
Unique image sizes: 1
Image formats: Counter({'JPEG': 7056})
First image size: (640, 640)

VALID
Images checked: 882
Unique image sizes: 1
Image formats: Counter({'JPEG': 882})
First image size: (640, 640)

TEST
Images checked: 872
Unique image sizes: 1
Image formats: Counter({'JPEG': 872})
First image size: (640, 640)


## 3.1 Aspect Ratio Analysis

All Dataset 2 images have dimensions of 640 × 640.

Therefore, all images have an aspect ratio of 1.0 and no additional aspect-ratio correction is required.

In [9]:
# Confirm the aspect ratio of Dataset 2 images

print("Image dimensions: 640 × 640")
print("Aspect ratio:", 640 / 640)

Image dimensions: 640 × 640
Aspect ratio: 1.0


# 4. Preprocessing Plan

Dataset 2 images are already standardized at 640 × 640 with a 1.0 aspect ratio.

Before model training, the images will be resized to the common input size used by the model and normalized.

The original Dataset 2 files will not be modified.

## 4.1 Create Processed Dataset Folders

Processed images will be saved separately from the raw dataset.

The train, validation, and test splits will be preserved.

In [11]:
# Create folders for the processed Dataset 2

PROCESSED_DATASET = PROJECT_ROOT / "data" / "processed" / "Dataset2"

for split in ["train", "valid", "test"]:
    (PROCESSED_DATASET / split / "images").mkdir(
        parents=True,
        exist_ok=True
    )

print("Processed dataset folders created:")
print(PROCESSED_DATASET)

Processed dataset folders created:
C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\Dataset2


## 4.2 Resize Images

Each image will be resized from 640 × 640 to 224 × 224.

The resized images will be saved in `data/processed/Dataset2/`.

The original images in `data/raw/Dataset/` will not be changed.

In [12]:
# Resize Dataset 2 images to 224 × 224

from PIL import Image

TARGET_SIZE = (224, 224)

for split in ["train", "valid", "test"]:
    
    input_dir = DATASET_ROOT / split
    output_dir = PROCESSED_DATASET / split / "images"
    
    # Find image files
    image_files = [
        f for f in input_dir.iterdir()
        if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
    ]
    
    processed = 0
    failed = 0
    
    for image_path in image_files:
        try:
            # Open image
            with Image.open(image_path) as img:
                
                # Convert to RGB
                img = img.convert("RGB")
                
                # Resize to 224 × 224
                img = img.resize(TARGET_SIZE)
                
                # Save as JPEG
                output_path = output_dir / f"{image_path.stem}.jpg"
                img.save(output_path, "JPEG")
                
                processed += 1
                
        except Exception as e:
            failed += 1
            print(f"Failed: {image_path.name} | {e}")
    
    print(f"\n{split.upper()}")
    print("Images processed:", processed)
    print("Images failed:", failed)

Failed: X-rays-of-a-three-year-old-girl-s-right-ankle-Anteromedial-undulation-of-the-tibial-physis-is-evident-and-the-fibular-physis-reaches-the-tibial-articular-surface-_png.rf.5066e38ff3e62bee36e6b76f7fcb4117.jpg | [Errno 2] No such file or directory: 'C:\\Users\\DELL\\Downloads\\mlproject\\bone-cancer-detection\\data\\raw\\Dataset\\train\\X-rays-of-a-three-year-old-girl-s-right-ankle-Anteromedial-undulation-of-the-tibial-physis-is-evident-and-the-fibular-physis-reaches-the-tibial-articular-surface-_png.rf.5066e38ff3e62bee36e6b76f7fcb4117.jpg'

TRAIN
Images processed: 7056
Images failed: 1

VALID
Images processed: 882
Images failed: 0

TEST
Images processed: 872
Images failed: 0


## 4.3 Verify Processed Images

We verify that the resized images were created correctly and have the expected dimensions.

In [13]:
# Verify the processed image dimensions

for split in ["train", "valid", "test"]:
    
    output_dir = PROCESSED_DATASET / split / "images"
    
    image_files = [
        f for f in output_dir.iterdir()
        if f.suffix.lower() == ".jpg"
    ]
    
    sizes = []
    
    for image_path in image_files:
        try:
            with Image.open(image_path) as img:
                sizes.append(img.size)
        except:
            pass
    
    print(f"\n{'=' * 50}")
    print(split.upper())
    print("=" * 50)
    print("Processed images:", len(sizes))
    print("Unique dimensions:", set(sizes))


TRAIN
Processed images: 7056
Unique dimensions: {(224, 224)}

VALID
Processed images: 882
Unique dimensions: {(224, 224)}

TEST
Processed images: 872
Unique dimensions: {(224, 224)}


# 5. Create Processed Metadata

Create metadata files linking each processed image to its binary cancer label.

Cancer = 1  
Normal = 0

The original train, validation, and test splits are preserved.

In [14]:
# Create metadata CSV files for the processed Dataset 2 images

for split in ["train", "valid", "test"]:

    # Path to the original labels CSV
    csv_path = DATASET_ROOT / split / "_classes.csv"

    # Path to the processed images
    processed_dir = PROCESSED_DATASET / split / "images"

    # Load the original labels
    labels = pd.read_csv(csv_path)

    # Remove accidental spaces from column names
    labels.columns = labels.columns.str.strip()

    # Create binary cancer label
    # Cancer = 1, Normal = 0
    labels["cancer_label"] = labels["cancer"].astype(int)

    # Convert original filename to processed filename
    labels["processed_filename"] = (
        labels["filename"]
        .astype(str)
        .apply(lambda x: Path(x).stem + ".jpg")
    )

    # Keep only images that were successfully processed
    labels = labels[
        labels["processed_filename"].apply(
            lambda x: (processed_dir / x).exists()
        )
    ].copy()

    # Keep only the columns needed for model training
    metadata = labels[
        ["processed_filename", "cancer_label"]
    ].copy()

    # Save metadata inside each processed split
    output_csv = PROCESSED_DATASET / split / "metadata.csv"

    metadata.to_csv(output_csv, index=False)

    print("=" * 50)
    print(split.upper())
    print("=" * 50)
    print("Metadata rows:", len(metadata))
    print("Cancer:", (metadata["cancer_label"] == 1).sum())
    print("Normal:", (metadata["cancer_label"] == 0).sum())
    print("Saved to:", output_csv)

TRAIN
Metadata rows: 7056
Cancer: 3081
Normal: 3975
Saved to: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\Dataset2\train\metadata.csv
VALID
Metadata rows: 882
Cancer: 398
Normal: 484
Saved to: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\Dataset2\valid\metadata.csv
TEST
Metadata rows: 872
Cancer: 384
Normal: 488
Saved to: C:\Users\DELL\Downloads\mlproject\bone-cancer-detection\data\processed\Dataset2\test\metadata.csv


In [15]:
# Final verification of processed images and metadata

for split in ["train", "valid", "test"]:

    processed_dir = PROCESSED_DATASET / split / "images"
    metadata_path = PROCESSED_DATASET / split / "metadata.csv"

    # Load processed metadata
    metadata = pd.read_csv(metadata_path)

    # Get processed image filenames
    image_files = {
        f.name for f in processed_dir.iterdir()
        if f.suffix.lower() == ".jpg"
    }

    # Get filenames recorded in metadata
    metadata_files = set(metadata["processed_filename"])

    # Compare images and metadata
    images_without_metadata = image_files - metadata_files
    metadata_without_images = metadata_files - image_files

    print(f"\n{'=' * 50}")
    print(split.upper())
    print("=" * 50)

    print("Processed images:", len(image_files))
    print("Metadata rows:", len(metadata))
    print("Images without metadata:", len(images_without_metadata))
    print("Metadata without images:", len(metadata_without_images))

    # Check dimensions
    sizes = []

    for filename in image_files:
        with Image.open(processed_dir / filename) as img:
            sizes.append(img.size)

    print("Image dimensions:", set(sizes))


TRAIN
Processed images: 7056
Metadata rows: 7056
Images without metadata: 0
Metadata without images: 0
Image dimensions: {(224, 224)}

VALID
Processed images: 882
Metadata rows: 882
Images without metadata: 0
Metadata without images: 0
Image dimensions: {(224, 224)}

TEST
Processed images: 872
Metadata rows: 872
Images without metadata: 0
Metadata without images: 0
Image dimensions: {(224, 224)}
